# Sıfırdan Dil Modeli · Ara Bölüm A
## PyTorch'a hızlı giriş

Modelimizi kuracağımız alet çantası: **PyTorch**. Bu defterde üç şeyle tanışıyoruz: **tensörler** (PyTorch'un sayı tabloları) ve şekilleri, **matris çarpımı**, ve **autograd**: PyTorch'un gradyanları kendisi hesaplaması. Sonunda üçünü kısa bir **eğitim döngüsünde** birleştiriyoruz: model, sekiz sıcaklıktan Celsius'u Fahrenheit'a çevirmeyi kendi kendine öğreniyor.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının A ekini (PyTorch'a giriş) izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. Tensör: PyTorch'un sayı tablosu
PyTorch Colab'da kurulu gelir; `import` onu deftere çağırır. Önce hangi sürümün kurulu olduğuna bakalım:

In [ ]:
import torch

print(torch.__version__)

**Tensör**, PyTorch'un sayı tablosu: tek bir sayı (skaler), bir sıra sayı (vektör), satırları ve sütunları olan bir tablo (matris) ya da tabloların yığını olabilir. `ndim` kaç boyutu olduğunu, `shape` her boyutta kaç sayı olduğunu söyler.

In [ ]:
tensor0d = torch.tensor(1)               # skaler: tek bir sayı
tensor1d = torch.tensor([1, 2, 3])       # vektör: bir sıra sayı
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])     # matris: satırlar, sütunlar
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])   # matrislerin yığını

for t in [tensor0d, tensor1d, tensor2d, tensor3d]:
    print(t.ndim, "boyut:", t.shape)

Her tensörün bir sayı türü var (`dtype`). Tam sayılardan kurulan tensör `int64`, ondalıklı sayılardan kurulan `float32` olur. Token numaraları tam sayı olacak; modelin ağırlıkları ise hep ondalıklı.

In [ ]:
print(tensor1d.dtype)

floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

## 2. Şekil
`reshape` aynı sayıları başka bir şekle sokar: sayıları sırayla okuyup yeni satırlara dizer. (`view` da aynı işi yapar; ileride ikisini de göreceğiz.)

In [ ]:
print(tensor2d)
print(tensor2d.reshape(3, 2))

`.T` ise **devriği** verir: satırları sütun yapar. Şekil yine 3 × 2, ama sayılar başka yerlerde.

In [ ]:
print(tensor2d.T)

## 3. Matris çarpımı
`@` işareti matris çarpımı. Birinci matrisin her satırı ikincisinin her sütunuyla eleman eleman çarpılıp toplanır; ilk hücre 1·1 + 2·2 + 3·3 = 14. (2 × 3) @ (3 × 2) → (2 × 2).

In [ ]:
print(tensor2d @ tensor2d.T)

Ortadaki boyutlar tutmalı: birincinin sütun sayısı, ikincinin satır sayısına eşit olmalı. Tutmazsa PyTorch hata verir (bu hücrenin hata vermesi beklenen bir şey):

In [ ]:
tensor2d @ tensor2d

Bir dil modelindeki hesabın çoğu bu işlem. Üç boyutlu bir tensörde `@` yalnız son iki boyuta bakar, öndeki boyut olduğu gibi kalır. `torch.ones` birlerle dolu bir tensör kurar; köşeli parantezdeki `[0, 0, :5]` her boyuttan bir seçim yapar: ilk tablo, ilk satır, ilk beş sayı.

In [ ]:
x = torch.ones(8, 4, 256)     # 8 tablo, her biri 4 satır × 256 sayı
W = torch.ones(256, 1024)     # 256 sayıyı 1.024 sayıya çeviren matris

out = x @ W
print(out.shape)
print(out[0, 0, :5])

## 4. Gradyan: autograd
Şimdi küçük bir model. Lise fiziğinden bir formül var: F = 1,8 · C + 32. Ama formülü modele **söylemeyeceğiz**; yalnızca aynı sekiz sıcaklığı iki ölçekte vereceğiz. `item()` bir tensörün içindeki sayıyı düz bir Python sayısı olarak verir.

In [ ]:
celsius = torch.tensor([-40.0, -10.0, 0.0, 10.0,
                        20.0, 30.0, 37.0, 100.0])
fahrenheit = torch.tensor([-40.0, 14.0, 32.0, 50.0,
                           68.0, 86.0, 98.6, 212.0])

for c, f in zip(celsius, fahrenheit):
    print(f"{c.item():6.1f} °C = {f.item():6.1f} °F")

Modelimiz iki sayı: **eğim** `w` ve **sabit** `b`; tahmini `w · C + b`. Başlangıçta `w = 1`, `b = 0`: model "Fahrenheit, Celsius'un aynısı" diyor. `requires_grad=True`, PyTorch'a bu sayıların gradyanını hesaplamasını söyler. Ne kadar yanıldığını **ortalama kare hata** ölçer: her farkın karesi, sonra ortalaması. Buna **kayıp** diyoruz.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)    # eğim
b = torch.tensor(0.0, requires_grad=True)    # sabit

pred = w * celsius + b                       # modelin tahmini
loss = ((pred - fahrenheit) ** 2).mean()     # ortalama kare hata
print(pred)
print(loss)

`grad_fn`, PyTorch'un bu sonucun nasıl hesaplandığını not ettiğini gösterir. `backward()` bu kaydı sondan başa izler ve kaybın her sayıya göre **gradyanını** hesaplar: o sayıyı biraz artırırsak kayıp ne kadar değişir? Sonuç `.grad`'a yazılır.

In [ ]:
loss.backward()

print(w.grad)
print(b.grad)

Gradyanı elle sınayalım: `w`'yi binde bir artırınca kayıp ne kadar değişiyor? `torch.no_grad()` altındaki hesaplar kaydedilmez; burada yalnızca bakıyoruz.

In [ ]:
with torch.no_grad():
    w_new = w + 0.001                        # w'yi binde bir artır
    loss_new = ((w_new * celsius + b - fahrenheit) ** 2).mean()

print(f"kayıptaki değişim: {loss_new.item() - loss.item():.4f}")
print(f"gradyan × 0,001:   {w.grad.item() * 0.001:.4f}")

## 5. Bir adım
Gradyan eksiyse sayıyı artırmak kaybı azaltır, artıysa azaltmak. Öyleyse her sayıyı gradyanının **tersine**, küçük bir adımla iteriz. Adımın boyu `lr` (öğrenme oranı). `-=` bir sayıdan çıkarıp sonucu yine ona yazar.

In [ ]:
lr = 0.0005                     # öğrenme oranı: adımın boyu
with torch.no_grad():
    w -= lr * w.grad            # gradyanın tersine bir adım
    b -= lr * b.grad
print(f"w = {w.item():.4f}   b = {b.item():.4f}")

loss = ((w * celsius + b - fahrenheit) ** 2).mean()
print(f"kayıp: {loss.item():.4f}")

Yeni kayıp için gradyanları yeniden hesaplayalım:

In [ ]:
loss.backward()

print(w.grad)

Dikkat: bu, yeni gradyan değil. PyTorch gradyanları **biriktirir**: yenisi eskisinin üstüne eklendi. Her adımdan önce eskileri sıfırlamak gerekir; `zero_()` bunu yapar.

In [ ]:
w.grad.zero_()                  # eski gradyanları sıfırla
b.grad.zero_()

loss = ((w * celsius + b - fahrenheit) ** 2).mean()
loss.backward()
print(w.grad)

## 6. Eğitim döngüsü
Bu adımları elle yapmak yerine bir **optimizer**'a bırakıyoruz: `zero_grad()` eski gradyanları siler, `step()` her sayıyı gradyanının tersine iter. `SGD` en sade optimizer: az önce elle yaptığımızın aynısı. Döngü her turda aynı beş işi yapar ve her bin adımda bir durumu yazdırır. `%` bölümden kalan: `step % 1000 == 0`, "adım binin katıysa".

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.0005)

for step in range(10001):
    pred = w * celsius + b                     # 1. tahmin et
    loss = ((pred - fahrenheit) ** 2).mean()   # 2. ne kadar yanıldık?
    if step % 1000 == 0:
        print(f"adım {step:5d}   kayıp {loss.item():9.4f}   "
              f"w {w.item():.4f}   b {b.item():.4f}")
    optimizer.zero_grad()                      # 3. eski gradyanları sil
    loss.backward()                            # 4. gradyanları hesapla
    optimizer.step()                           # 5. bir adım at

Model formülü kendisi buldu. Hiç görmediği bir sıcaklıkla deneyelim:

In [ ]:
with torch.no_grad():
    print(w * 25 + b)           # 25 °C kaç °F?

## 7. Gerçek boy
`SGD` her sayıya aynı boyda adım attırır; `b` gibi yavaş ilerleyenler on bin adım bekler. **Adam** her sayının adım boyunu kendisi ayarlar. Döngünün geri kalanı aynı; yalnızca optimizer satırı değişiyor.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.Adam([w, b], lr=0.3)   # tek değişen satır

for step in range(501):
    pred = w * celsius + b
    loss = ((pred - fahrenheit) ** 2).mean()
    if step % 50 == 0:
        print(f"adım {step:5d}   kayıp {loss.item():9.4f}   "
              f"w {w.item():.4f}   b {b.item():.4f}")
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

Serinin modeli **Çırak** da tam bu döngüyle eğitildi: iki sayı yerine yaklaşık 5,86 milyon sayı, sekiz sıcaklık yerine Türkçe Vikipedi, `Adam` yerine onun yakın akrabası `AdamW`, 10.000 adım. Kitabın 5. bölümündeki eğitim döngüsü de aynı üç satırı kullanır: `zero_grad()`, `backward()`, `step()`.

## Özet
**Tensör** PyTorch'un sayı tablosu; `shape` şeklini, `dtype` sayı türünü söyler. `reshape` sayıları yeniden dizer, `.T` satırları sütun yapar. `@` matris çarpımı: ortadaki boyutlar tutmalı. `requires_grad=True` olan sayıların gradyanını `backward()` hesaplar; gradyan, sayıyı hangi yöne itersek kaybın azalacağını söyler. Eğitim döngüsü: tahmin et, kaybı ölç, gradyanları sıfırla, hesapla, bir adım at; tekrarla.

**Kendin dene**
- Eğitim döngüsünde `lr`'yi `0.001` yapın: kayıp neden patlıyor?
- `fahrenheit` yerine Kelvin verin (K = C + 273,15): model hangi iki sayıyı buluyor?

**Sıradaki bölüm:** kayan pencere, öyküden girdi ve hedef çiftleri.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, Ek A (PyTorch'a giriş). Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli.